# Aswini Speaker Diarization Tool
Separates Aswini's voice from call recordings (WhatsApp MPEG files).

**Before running:**
1. Add your dataset: `lokeshk431/calles` → appears at `/kaggle/input/calles/`
2. Add HF secret: Secrets → Add → Name: `HF_TOKEN`, Value: your HuggingFace token
3. Enable Internet in notebook settings
4. Runtime: GPU T4
5. Run cells 1 → 8 in order

In [ ]:
# CELL 1 — Install (ignore the RAPIDS/numba conflict warnings, they are harmless)
!pip install -q pyannote.audio pydub openai-whisper

In [ ]:
# CELL 2 — Paths + HF Token
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

HF_TOKEN   = UserSecretsClient().get_secret('HF_TOKEN')

# Dataset path on Kaggle filesystem (NOT the URL path)
# URL: kaggle.com/datasets/lokeshk431/calles → filesystem: /kaggle/input/calles/
INPUT_DIR  = Path('/kaggle/input/calles')
WORK_DIR   = Path('/kaggle/working')
WAV_DIR    = WORK_DIR / 'wavs_converted'
OUT_DIR    = WORK_DIR / 'aswini_clips'

WAV_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {'.mpeg', '.mpg', '.wav', '.ogg', '.opus', '.m4a', '.mp3', '.flac'}
audio_files = sorted(p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in AUDIO_EXT)

print(f'Found {len(audio_files)} audio files:')
for p in audio_files:
    print(' ', p.name)

In [ ]:
# CELL 3 — Convert all MPEG → WAV (16kHz mono, required by pyannote)
import subprocess
from tqdm.auto import tqdm

converted = []
for src in tqdm(audio_files, desc='Converting'):
    dst = WAV_DIR / (src.stem + '.wav')
    if not dst.exists():
        subprocess.run(
            ['ffmpeg', '-y', '-i', str(src), '-ar', '16000', '-ac', '1', str(dst)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    if dst.exists():
        converted.append(dst)
    else:
        print(f'  FAILED: {src.name}')

print(f'Converted: {len(converted)} files')

In [ ]:
# CELL 4 — Load pyannote diarization pipeline
# Requires accepting terms at:
#   https://huggingface.co/pyannote/speaker-diarization-3.1
#   https://huggingface.co/pyannote/segmentation-3.0
from pyannote.audio import Pipeline
import torch

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    token=HF_TOKEN
)
pipeline = pipeline.to(torch.device('cuda'))
print('Pipeline ready on', next(pipeline.parameters()).device)

In [ ]:
# CELL 5 — Diarize all calls and identify Aswini
# Strategy: Aswini is the OUTBOUND caller.
#   - She speaks first AND has the most total speaking time.
#   - We pick the speaker with the longest cumulative duration.
import json
from tqdm.auto import tqdm

MIN_CLIP_S = 2.5
MAX_CLIP_S = 8.0

all_segments = []  # will hold (wav_stem, start, end, text_placeholder)
diarization_log = {}

for wav in tqdm(converted, desc='Diarizing'):
    try:
        diarization = pipeline(str(wav))
    except Exception as e:
        print(f'  SKIP {wav.name}: {e}')
        continue

    # Accumulate duration per speaker
    speaker_dur = {}
    segments_by_speaker = {}
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        dur = turn.end - turn.start
        speaker_dur[speaker] = speaker_dur.get(speaker, 0) + dur
        segments_by_speaker.setdefault(speaker, []).append((turn.start, turn.end))

    if not speaker_dur:
        print(f'  No speech detected: {wav.name}')
        continue

    # Aswini = speaker with MOST total speaking time
    aswini_label = max(speaker_dur, key=speaker_dur.get)
    aswini_dur   = speaker_dur[aswini_label]
    total_dur    = sum(speaker_dur.values())

    print(f'  {wav.name}: {len(speaker_dur)} speakers, '
          f'Aswini={aswini_label} ({aswini_dur:.1f}s / {total_dur:.1f}s total)')

    diarization_log[wav.stem] = {
        'aswini_label': aswini_label,
        'speaker_durations': speaker_dur
    }

    for start, end in segments_by_speaker[aswini_label]:
        dur = end - start
        if MIN_CLIP_S <= dur <= MAX_CLIP_S:
            all_segments.append({
                'source': wav.stem,
                'start': round(start, 3),
                'end':   round(end,   3),
                'duration': round(dur, 3)
            })

print(f'\nTotal Aswini segments ({MIN_CLIP_S}-{MAX_CLIP_S}s): {len(all_segments)}')

# Save diarization log
with open(WORK_DIR / 'diarization_log.json', 'w') as f:
    json.dump(diarization_log, f, indent=2)

In [ ]:
# CELL 6 — Export Aswini clips as WAV files
from pydub import AudioSegment as AS
from tqdm.auto import tqdm

PADDING_MS = 100  # add 100ms silence padding each side
wav_cache  = {}

exported = []
for i, seg in enumerate(tqdm(all_segments, desc='Exporting')):
    src_path = WAV_DIR / (seg['source'] + '.wav')
    if str(src_path) not in wav_cache:
        wav_cache[str(src_path)] = AS.from_wav(str(src_path))
    audio = wav_cache[str(src_path)]

    start_ms = max(0, int(seg['start'] * 1000) - PADDING_MS)
    end_ms   = min(len(audio), int(seg['end']   * 1000) + PADDING_MS)
    clip     = audio[start_ms:end_ms]

    out_name = f'aswini_{i:04d}.wav'
    out_path = OUT_DIR / out_name
    clip.export(str(out_path), format='wav')

    exported.append({
        'audio_path': str(out_path),
        'source': seg['source'],
        'start': seg['start'],
        'end': seg['end'],
        'duration': seg['duration'],
        'text': ''  # filled by Cell 7
    })

print(f'Exported {len(exported)} clips to {OUT_DIR}')

In [ ]:
# CELL 7 — Transcribe Aswini clips with Whisper
import whisper
from tqdm.auto import tqdm

model = whisper.load_model('medium')  # use 'small' if this is slow
print('Whisper medium loaded')

for item in tqdm(exported, desc='Transcribing'):
    try:
        result = model.transcribe(
            item['audio_path'],
            language='en',
            condition_on_previous_text=False
        )
        item['text'] = result['text'].strip()
    except Exception as e:
        item['text'] = ''
        print(f"  {item['audio_path']}: {e}")

# Drop empty transcripts
exported = [x for x in exported if x['text']]
print(f'With transcripts: {len(exported)}')

In [ ]:
# CELL 8 — Save dataset.jsonl + summary
import json

jsonl_path = WORK_DIR / 'aswini_dataset.jsonl'
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for item in exported:
        f.write(json.dumps(item) + '\n')

total_sec = sum(x['duration'] for x in exported)
print(f'Dataset saved: {jsonl_path}')
print(f'Clips  : {len(exported)}')
print(f'Total  : {total_sec/60:.1f} minutes of Aswini speech')
print()
print('--- Preview (first 5) ---')
for item in exported[:5]:
    print(f"  [{item['duration']:.1f}s] {item['text']}")
print()
print('Download from Kaggle Output tab:')
print('  /kaggle/working/aswini_clips/   (WAV files)')
print('  /kaggle/working/aswini_dataset.jsonl')